# MMA3001 — Indoor Temperature Analysis

**Clean seven-section notebook, reorganised from `Untitled0.ipynb` (28 original code cells).**

This notebook retains the original temperature JSON extraction, selected sensor, chronological split, NumPy polynomial models, fixed piecewise boundaries, validation/test metrics, sensitivity study, trapezoidal integration and error/sampling checks. Run cells **top to bottom** in Google Colab. The original notebook is not modified.

**Input:** `5EnvSensor_MayToDec2024_180kRows.csv` (or a ZIP containing it). The first cell tries a local file, then the GitHub repository, then Colab upload. **Outputs:** tables and figures calculated from the actual data, not hard-coded results.

## 1 — Data loading and temperature extraction

**Purpose.** Import the raw environmental-sensor CSV and unpack the nested JSON measurements.

**Method.** In each `jsondata` record, the original dataset stores a list of variables. We select `item["variable"]["name"] == "Temperature"` and then read each entry in `item["values"]`. The Unix timestamp is converted to Melbourne local time. Invalid or duplicate records are discarded.

**Check the output.** The raw dataset should have columns such as `sensorid` and `jsondata`; the cleaned table should have `sensorid`, `timestamp`, `temperature` and `datetime`.

In [ ]:
import io
import json
import zipfile
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

REPO = "Nervermind13/MMA3001-Indoor-Temperature"
DATASET_HINT = "5EnvSensor_MayToDec2024_180kRows"

def read_dataset(data, filename):
    """Read a CSV, or the first matching CSV stored inside a ZIP."""
    if filename.lower().endswith(".csv"):
        return pd.read_csv(io.BytesIO(data))
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        names = [name for name in archive.namelist() if name.lower().endswith(".csv")]
        if not names:
            raise ValueError("ZIP contains no CSV")
        match = [name for name in names if DATASET_HINT.lower() in name.lower()]
        chosen = (match or names)[0]
        print("CSV inside ZIP:", chosen)
        with archive.open(chosen) as f:
            return pd.read_csv(f)

def load_github_dataset():
    """Find the dataset filename in the public GitHub repository; do not guess it."""
    api = f"https://api.github.com/repos/{REPO}"
    info = requests.get(api, timeout=30)
    info.raise_for_status()
    branch = info.json()["default_branch"]
    response = requests.get(f"{api}/git/trees/{branch}", params={"recursive": 1}, timeout=30)
    response.raise_for_status()
    candidates = [item["path"] for item in response.json()["tree"]
                  if item["type"] == "blob" and item["path"].lower().endswith((".csv", ".zip"))]
    print("GitHub dataset candidates:", candidates)
    matching = [name for name in candidates if DATASET_HINT.lower() in name.lower()]
    if not matching:
        raise FileNotFoundError("Dataset not found in GitHub repository")
    filename = matching[0]
    raw = f"https://raw.githubusercontent.com/{REPO}/{quote(branch, safe='')}/{quote(filename, safe='/')}"
    download = requests.get(raw, timeout=180)
    download.raise_for_status()
    return read_dataset(download.content, filename)

local_files = [Path("5EnvSensor_MayToDec2024_180kRows.csv"),
               Path("/content/5EnvSensor_MayToDec2024_180kRows.csv")]
local_file = next((p for p in local_files if p.exists()), None)

if local_file is not None:
    print("Loading local dataset:", local_file)
    df = pd.read_csv(local_file)
else:
    try:
        df = load_github_dataset()
    except Exception as error:
        print("GitHub loading unavailable:", error)
        from google.colab import files
        print("Upload the lecturer's sensor CSV or ZIP:")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No dataset uploaded")
        filename = next(iter(uploaded))
        df = read_dataset(uploaded[filename], filename)

print("Raw dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(3))

In [ ]:
# Original notebook's nested JSON extraction, preserved.
temperature_records = []
for _, row in df.iterrows():
    try:
        measurements = json.loads(row["jsondata"])
        for item in measurements:
            if item["variable"]["name"] == "Temperature":
                for reading in item["values"]:
                    temperature_records.append([row["sensorid"], reading["time"], reading["value"]])
    except (ValueError, TypeError, KeyError):
        continue

temp_df = pd.DataFrame(temperature_records,
                       columns=["sensorid", "timestamp", "temperature"])
if temp_df.empty:
    raise ValueError("No temperature data extracted; check the JSON format")
temp_df["temperature"] = pd.to_numeric(temp_df["temperature"], errors="coerce")
temp_df["datetime"] = pd.to_datetime(temp_df["timestamp"], unit="s", utc=True,
                                     errors="coerce").dt.tz_convert("Australia/Melbourne")
temp_df = temp_df.dropna(subset=["sensorid", "datetime", "temperature"])
temp_df = temp_df.drop_duplicates(subset=["sensorid", "timestamp"])
temp_df = temp_df.sort_values("datetime")
print("Temperature readings:", len(temp_df))
print("Sensors:", temp_df["sensorid"].nunique())
display(temp_df.head())

**Interpretation.** One raw CSV row can contain several measurements. The number of extracted temperature readings is therefore not necessarily equal to the number of CSV rows. The timezone conversion ensures that the hour-of-day analysis reflects local Melbourne time.

## 2 — Data cleaning, sensor selection and chronological split

**Purpose.** Keep the selected indoor sensor, identify days with enough data, and separate training, validation and test days.

**Method.** Preserve the original rules: at least **120 readings per day** and readings covering **all 24 hours**. The original notebook uses `int(n*0.70)` and `int(n*0.85)` to set chronological split boundaries. No test observations are used when fitting the models.

In [ ]:
sensor_id = 6012002000869  # Original selected sensor
sensor_data = temp_df[temp_df["sensorid"] == sensor_id].copy()
sensor_data = sensor_data[(sensor_data["datetime"] >= "2024-05-01") &
                          (sensor_data["datetime"] < "2025-01-01")].copy()
sensor_data["date"] = sensor_data["datetime"].dt.date
sensor_data["hour"] = sensor_data["datetime"].dt.hour

if sensor_data.empty:
    raise ValueError(f"No temperature readings found for sensor {sensor_id}")

daily_counts = sensor_data.groupby("date").size()
valid_days = daily_counts[daily_counts >= 120].index
clean_data = sensor_data[sensor_data["date"].isin(valid_days)].copy()
hourly_coverage = clean_data.groupby("date")["hour"].nunique()
complete_days = hourly_coverage[hourly_coverage == 24].index
clean_data = clean_data[clean_data["date"].isin(complete_days)].copy()
dates = sorted(clean_data["date"].unique())
n = len(dates)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
train_dates = dates[:train_end]
val_dates = dates[train_end:val_end]
test_dates = dates[val_end:]
if not train_dates or not val_dates or not test_dates:
    raise ValueError("Insufficient complete days for train/validation/test split")
train_data = clean_data[clean_data["date"].isin(train_dates)].copy()
val_data = clean_data[clean_data["date"].isin(val_dates)].copy()
test_data = clean_data[clean_data["date"].isin(test_dates)].copy()

print("Selected sensor:", sensor_id)
print("Total selected readings:", len(sensor_data))
print("Complete days:", n)
for label, days in [("Training", train_dates), ("Validation", val_dates), ("Testing", test_dates)]:
    print(f"{label}: {len(days)} days ({days[0]} to {days[-1]})")
print("Temperature summary (°C):")
display(clean_data["temperature"].describe())

**Interpretation.** Keeping complete days reduces the chance that missing hours distort the average daily pattern. Splitting by date instead of randomly splitting individual readings provides a more realistic evaluation on later days.

## 3 — Exploratory temperature and sampling analysis

**Purpose.** Observe temperature variation, the mean daily cycle and sensor sampling intervals.

**Method.** Plot selected temperature readings over time, average temperature for each training hour, daily coverage, and within-day sampling intervals. Using training data for the average profile avoids relying on test data when building the model.

In [ ]:
train_hourly = train_data.groupby("hour")["temperature"].mean().reindex(range(24))
val_hourly = val_data.groupby("hour")["temperature"].mean().reindex(range(24))
test_hourly = test_data.groupby("hour")["temperature"].mean().reindex(range(24))

fig, axes = plt.subplots(3, 1, figsize=(11, 11))
axes[0].plot(sensor_data["datetime"], sensor_data["temperature"], linewidth=0.7)
axes[0].set(title=f"Temperature readings — sensor {sensor_id}", ylabel="Temperature (°C)")
axes[1].plot(range(24), train_hourly.values, "o-")
axes[1].set(title="Training hourly mean temperature", xlabel="Hour", ylabel="Temperature (°C)")
axes[1].set_xticks(range(0, 24, 2))
axes[2].plot(daily_counts.index, daily_counts.values, marker=".")
axes[2].axhline(120, color="red", linestyle="--", label="Minimum 120 readings")
axes[2].set(title="Daily measurement coverage", xlabel="Date", ylabel="Readings per day")
axes[2].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

sampling_data = clean_data.sort_values("datetime").copy()
sampling_data["Interval (minutes)"] = (sampling_data.groupby("date")["datetime"]
                                         .diff().dt.total_seconds() / 60)
intervals = sampling_data["Interval (minutes)"].dropna()
print("Sampling intervals (minutes):")
print("Median:", round(intervals.median(), 2), "Mean:", round(intervals.mean(), 2))
print("Min:", round(intervals.min(), 2), "Max:", round(intervals.max(), 2))
print("Over 15 minutes:", int((intervals > 15).sum()))
plt.figure(figsize=(9, 4))
plt.hist(intervals, bins=np.arange(0, 35, 1), edgecolor="black")
plt.axvline(10, color="red", linestyle="--", label="Nominal 10-minute interval")
plt.xlabel("Minutes between readings")
plt.ylabel("Number of intervals")
plt.title("Sensor sampling interval distribution")
plt.legend()
plt.tight_layout()
plt.show()

**Interpretation.** Check whether the mean profile cools overnight and warms in the morning. The time intervals show whether readings are evenly spaced. These plots describe measured behaviour; they do not establish what caused temperature changes.

## 4 — Numerical modelling: linear, cubic and piecewise regression

**Purpose.** Represent the training-hour temperature profile using three simple models.

**Method.** NumPy's `polyfit` finds least-squares polynomial coefficients. Linear regression uses degree 1, cubic regression degree 3, and the piecewise model fits three independent straight lines over **0–6, 6–8 and 8–23 hours**. The original notebook fits both sides of a boundary and lets the later segment overwrite its shared-hour prediction; this behaviour is preserved for comparable results.

The least-squares objective is \(\min \sum_i (T_i - \hat{T}(h_i))^2\). Here \(T_i\) is the training hourly mean and \(\hat{T}(h_i)\) is the fitted value at hour \(h_i\).

In [ ]:
hours = np.arange(24)
if train_hourly.isna().any():
    raise ValueError("A training hour has no temperature readings")
linear_fit = np.poly1d(np.polyfit(hours, train_hourly.values, 1))
cubic_fit = np.poly1d(np.polyfit(hours, train_hourly.values, 3))

def fit_piecewise(hourly_values, b1=6, b2=8):
    """Return 24 predictions from three independent least-squares lines."""
    predictions = np.zeros(24)
    for start, end in [(0, b1), (b1, b2), (b2, 23)]:
        mask = (hours >= start) & (hours <= end)
        coeff = np.polyfit(hours[mask], np.asarray(hourly_values)[mask], 1)
        predictions[mask] = np.polyval(coeff, hours[mask])
    return predictions

piecewise_predictions = fit_piecewise(train_hourly.values, 6, 8)
plt.figure(figsize=(10, 5))
plt.plot(hours, train_hourly.values, "ko", label="Training hourly means")
plt.plot(hours, linear_fit(hours), label="Linear")
plt.plot(hours, cubic_fit(hours), label="Cubic")
plt.plot(hours, piecewise_predictions, label="Piecewise (06:00, 08:00)")
plt.xlabel("Hour of day")
plt.ylabel("Temperature (°C)")
plt.title("Regression models fitted to training data")
plt.xticks(range(0, 24, 2))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretation.** A single line cannot describe a strong morning transition and stable daytime behaviour simultaneously. The cubic model can bend smoothly. The piecewise model allows separate rates of change but may introduce jumps at the two boundaries.

## 5 — Validation, breakpoint sensitivity and untouched test

**Purpose.** Measure prediction accuracy on later days, check how breakpoint choices affect error, and report performance on the final unseen test period.

**Metrics.** \(\mathrm{RMSE}=\sqrt{rac{1}{N}\sum_i(\hat{T}_i-T_i)^2}\); \(\mathrm{MAE}=rac{1}{N}\sum_i|\hat{T}_i-T_i|\). Both have units of °C. Lower values are better.

**Original-method choice.** The main piecewise model retains **06:00 and 08:00**. Sensitivity compares alternative breakpoints on validation *hourly means*, as in the original notebook; it does not silently change the final model. We also report individual-reading errors, which are larger than hourly-mean errors because individual measurements vary within an hour.

In [ ]:
def calculate_errors(actual, predicted):
    """Return RMSE and MAE, both in degrees Celsius."""
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    return (float(np.sqrt(np.mean((actual - predicted)**2))),
            float(np.mean(np.abs(actual - predicted))))

def model_comparison(data, title):
    """Evaluate three already-fitted models on individual sensor readings."""
    h = data["hour"].to_numpy()
    actual = data["temperature"].to_numpy()
    predictions = [("Linear", linear_fit(h)), ("Cubic", cubic_fit(h)),
                   ("Piecewise", piecewise_predictions[h])]
    rows = []
    for name, predicted in predictions:
        rmse, mae = calculate_errors(actual, predicted)
        rows.append([name, rmse, mae])
    result = pd.DataFrame(rows, columns=["Model", "RMSE (°C)", "MAE (°C)"])
    print(title)
    display(result.round(6))
    return result

validation_hourly = pd.DataFrame({
    "Model": ["Linear", "Cubic", "Piecewise"],
    "Hourly validation RMSE (°C)": [
        calculate_errors(val_hourly.values, linear_fit(hours))[0],
        calculate_errors(val_hourly.values, cubic_fit(hours))[0],
        calculate_errors(val_hourly.values, piecewise_predictions)[0]]
})
print("Validation on 24 hourly averages:")
display(validation_hourly.round(6))
validation_readings = model_comparison(val_data, "Validation on individual readings:")

boundary_options = [(5, 7), (5, 8), (6, 8), (6, 9), (7, 9)]
sensitivity_rows = []
for b1, b2 in boundary_options:
    predicted = fit_piecewise(train_hourly.values, b1, b2)
    score = calculate_errors(val_hourly.values, predicted)[0]
    sensitivity_rows.append([f"{b1:02d}:00", f"{b2:02d}:00", score])
sensitivity_results = pd.DataFrame(
    sensitivity_rows, columns=["Start", "End", "Validation hourly RMSE (°C)"]
).sort_values("Validation hourly RMSE (°C)")
print("Breakpoint sensitivity:")
display(sensitivity_results.round(6))

# The final test is evaluated only after model definition and validation.
test_results = model_comparison(test_data, "Final test on individual readings:")
plt.figure(figsize=(10, 5))
plt.plot(hours, test_hourly.values, "ko-", label="Actual test hourly mean")
plt.plot(hours, linear_fit(hours), "--", label="Linear")
plt.plot(hours, cubic_fit(hours), "--", label="Cubic")
plt.plot(hours, piecewise_predictions, label="Piecewise")
plt.xlabel("Hour of day")
plt.ylabel("Temperature (°C)")
plt.title("Final test: measured hourly means versus models")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Original steps 24–26: prediction bias, largest errors and time resolution.
actual = test_data["temperature"].to_numpy()
test_hours = test_data["hour"].to_numpy()
model_predictions = {"Linear": linear_fit(test_hours),
                     "Cubic": cubic_fit(test_hours),
                     "Piecewise": piecewise_predictions[test_hours]}
bias_rows = []
for name, predicted in model_predictions.items():
    error = predicted - actual
    bias_rows.append([name, np.mean(error), np.max(np.abs(error))])
bias_table = pd.DataFrame(bias_rows, columns=["Model", "Mean error (°C)",
                                                  "Maximum absolute error (°C)"])
print("Prediction bias (prediction minus actual):")
display(bias_table.round(4))

error_data = test_data.copy()
error_data["Predicted (°C)"] = piecewise_predictions[error_data["hour"].to_numpy()]
error_data["Error (°C)"] = error_data["Predicted (°C)"] - error_data["temperature"]
error_data["Absolute error (°C)"] = error_data["Error (°C)"].abs()
error_data["Decimal hour"] = (error_data["datetime"].dt.hour +
                               error_data["datetime"].dt.minute / 60 +
                               error_data["datetime"].dt.second / 3600)
print("Largest piecewise prediction errors:")
display(error_data.nlargest(10, "Absolute error (°C)")[
    ["datetime", "hour", "Decimal hour", "temperature", "Predicted (°C)",
     "Error (°C)", "Absolute error (°C)"]].round(4))

**Interpretation.** Use the **test** table for final performance claims. A negative mean prediction error means the model tends to underestimate temperature. The largest errors and decimal-hour values help explain why an integer-hour model misses rapid changes within a single hour. The sensitivity table describes how the piecewise model depends on assumed transition times.

## 6 — Trapezoidal numerical integration and verification

**Purpose.** Check whether uneven measurement spacing changes daily mean temperature estimates.

**Method.** The trapezoidal rule approximates the temperature-time area as \(Approx\sum_irac{T_i+T_{i+1}}{2}(t_{i+1}-t_i)\). Dividing by \(t_{last}-t_{first}\) gives the **time-weighted mean over the observed span**, which is compared with the arithmetic mean of the readings.

**Verification.** For a constant-temperature example, the time-weighted mean must equal the constant value. This is a basic independent check of the calculation.

In [ ]:
def time_weighted_average(times_hours, temperatures):
    """Trapezoidal mean temperature over the measured time span."""
    duration = times_hours[-1] - times_hours[0]
    if duration <= 0:
        raise ValueError("Need at least two distinct measurement times")
    return np.trapezoid(temperatures, times_hours) / duration

# Simple verification with unevenly spaced times.
assert np.isclose(time_weighted_average(np.array([0., 1., 3.]),
                                        np.array([22., 22., 22.])), 22.0)
print("Constant-temperature integration check passed.")

daily_results = []
for date, day_data in clean_data.groupby("date"):
    day_data = day_data.sort_values("datetime")
    time_hours = ((day_data["datetime"] - day_data["datetime"].iloc[0])
                  .dt.total_seconds().to_numpy() / 3600)
    temperatures = day_data["temperature"].to_numpy()
    weighted = time_weighted_average(time_hours, temperatures)
    ordinary = float(np.mean(temperatures))
    daily_results.append([date, weighted, ordinary, weighted - ordinary])

comparison = pd.DataFrame(daily_results,
    columns=["Date", "Time-weighted average (°C)", "Ordinary average (°C)",
             "Difference (°C)"])
print("Daily integration comparison:")
display(comparison.head(10).round(4))
print("Mean absolute difference (°C):", round(comparison["Difference (°C)"].abs().mean(), 6))
print("Maximum absolute difference (°C):", round(comparison["Difference (°C)"].abs().max(), 6))

**Interpretation.** A small difference indicates that ordinary averaging is adequate for these observations, despite occasional sampling gaps. The calculation is **not** an exact midnight-to-midnight mean because the first and last readings may not fall at day boundaries.

## 7 — Final results and engineering interpretation

**Purpose.** Collect the important outcomes in one place and save files for the report.

**Engineering meaning.** The models approximate a repeatable daily indoor temperature pattern; the best-performing model is the one with the lowest error on the untouched test days. This is not evidence of HVAC energy savings or proof of why temperatures change.

**Limitations.** Only one sensor and a limited 2024 date range are analysed; the models use integer hour of day rather than exact time, outdoor conditions or HVAC status. Piecewise regression may be discontinuous at its boundaries.

**AI transparency.** The original analysis was reorganised into seven sections with assistance. Verify the final calculations, explain your own model decisions, and disclose AI assistance according to the unit rules.

In [ ]:
print("FINAL TEST MODEL COMPARISON")
display(test_results.round(6))
print("VALIDATION BREAKPOINT SENSITIVITY")
display(sensitivity_results.round(6))
print("INTEGRATION COMPARISON")
print("Mean absolute difference (°C):", round(comparison["Difference (°C)"].abs().mean(), 6))

plt.figure(figsize=(8, 4))
plt.bar(test_results["Model"], test_results["RMSE (°C)"])
plt.ylabel("Test RMSE (°C)")
plt.title("Final comparison: error on unseen individual readings")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

from pathlib import Path
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)
test_results.to_csv(output_dir / "test_results.csv", index=False)
sensitivity_results.to_csv(output_dir / "breakpoint_sensitivity.csv", index=False)
comparison.to_csv(output_dir / "integration_comparison.csv", index=False)
bias_table.to_csv(output_dir / "prediction_bias.csv", index=False)
print("Saved CSV tables in results/.")

### Checklist for GitHub submission

- This notebook documents the **analysis**, but the assessment may also require a separate Python module with docstrings, automated tests, generated HTML documentation, a README, licence and test report.
- Confirm the final notebook runs top-to-bottom against the uploaded dataset.
- Save figures or regenerate them when preparing the written report.
- Keep the original `Untitled0.ipynb` as evidence of your exploratory work.